# GCN Example 3 — Link Prediction (Graph Auto-Encoder)

The third canonical graph task: instead of labelling nodes or graphs, we predict **which
edges exist**. Hide some of the Karate Club's friendships, then ask the model to recover
them. This powers friend/▶product recommendation and knowledge-graph completion.

## The Graph Auto-Encoder (GAE) recipe (Kipf & Welling, 2016)

1. **Encode** every node into a vector with a GCN: $Z = \text{GCN}(X, \hat{A})$.
2. **Decode** the probability of an edge between nodes $i$ and $j$ as the similarity of
   their vectors: $\hat{A}_{ij} = \sigma(z_i^\top z_j)$. Similar embeddings ⇒ likely edge.

## How we train & evaluate it

- **Split**: hold out 15% of the real edges as a *test* set. The GCN only sees the
  remaining 85% (the *message-passing* graph) — it must never peek at the answers.
- **Positives vs negatives**: each step we contrast real edges (label 1) against randomly
  sampled **non-edges** (label 0), with binary cross-entropy.
- **Metric**: **AUC** (area under the ROC curve) on held-out positive edges vs. sampled
  negative edges — the probability the model ranks a true edge above a fake one.

> Self-contained, runs in seconds on CPU. Tested with `tensorflow==2.1`.

In [ ]:
import numpy as np
import tensorflow as tf

np.random.seed(0)
tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. The graph and the train/test edge split

Same 78-edge Karate Club graph. We shuffle the edges and peel off 15% as test positives;
the message-passing adjacency is built from the **training edges only**.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]
N = 34
all_edges = [(min(i, j), max(i, j)) for i, j in edges]
edge_set = set(all_edges)            # every real edge, used to avoid sampling them as negatives

order = np.random.permutation(len(all_edges))
n_test = int(0.15 * len(all_edges))
test_pos = [all_edges[i] for i in order[:n_test]]
train_pos = [all_edges[i] for i in order[n_test:]]

# Message-passing adjacency from TRAIN edges only
A = np.zeros((N, N), dtype='float32')
for i, j in train_pos:
    A[i, j] = A[j, i] = 1.0

A_tilde = A + np.eye(N, dtype='float32')
d_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(1)))
A_hat = tf.constant((d_inv_sqrt @ A_tilde @ d_inv_sqrt).astype('float32'))

print(f'edges: {len(all_edges)} | train: {len(train_pos)} | test: {len(test_pos)}')

## 2. Negative sampling and the AUC metric

A **negative** is a node pair that is *not* a real edge. We never sample a true edge
(train or test) as a negative. AUC is computed from ranks — no `sklearn` needed.

In [ ]:
def sample_negatives(k):
    negs, seen = [], set()
    while len(negs) < k:
        i, j = np.random.randint(N), np.random.randint(N)
        if i == j:
            continue
        a, b = min(i, j), max(i, j)
        if (a, b) in edge_set or (a, b) in seen:
            continue
        seen.add((a, b)); negs.append((a, b))
    return negs

def auc_score(pos_scores, neg_scores):
    s = np.concatenate([pos_scores, neg_scores])
    order = np.argsort(s)
    ranks = np.empty(len(s)); ranks[order] = np.arange(1, len(s) + 1)
    n_pos, n_neg = len(pos_scores), len(neg_scores)
    return (ranks[:n_pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

## 3. The GCN encoder + dot-product decoder

A 2-layer GCN (`34 → 32 → 16`) maps the identity features to a 16-dim embedding per node.
The decoder scores an edge as the dot product of the two endpoint embeddings (a raw logit;
a sigmoid turns it into a probability).

In [ ]:
X = tf.eye(N)
init = tf.keras.initializers.GlorotUniform(seed=0)
W1 = tf.Variable(init((N, 32)))
W2 = tf.Variable(init((32, 16)))
variables = [W1, W2]

def encode():
    h = tf.nn.relu(A_hat @ (X @ W1))
    z = A_hat @ (h @ W2)
    return z

def edge_logits(z, pairs):
    pairs = np.asarray(pairs)
    zi = tf.gather(z, pairs[:, 0])
    zj = tf.gather(z, pairs[:, 1])
    return tf.reduce_sum(zi * zj, axis=1)   # dot-product decoder

## 4. Train

Each epoch we draw a fresh batch of negatives (as many as we have positives), score both,
and minimise binary cross-entropy. Resampling negatives every epoch keeps the model from
memorising a fixed set of non-edges.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(0.01)
train_pos_arr = np.array(train_pos)

EPOCHS = 200
for epoch in range(1, EPOCHS + 1):
    neg = sample_negatives(len(train_pos))
    with tf.GradientTape() as tape:
        z = encode()
        pos_logit = edge_logits(z, train_pos_arr)
        neg_logit = edge_logits(z, neg)
        loss = (bce(tf.ones_like(pos_logit), pos_logit) +
                bce(tf.zeros_like(neg_logit), neg_logit))
    grads = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(grads, variables))

    if epoch % 20 == 0 or epoch == 1:
        z = encode()
        test_neg = sample_negatives(len(test_pos))
        pos_s = edge_logits(z, test_pos).numpy()
        neg_s = edge_logits(z, test_neg).numpy()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={float(loss):.4f}  test_AUC={auc_score(pos_s, neg_s):.3f}')

## 5. Inspect predictions

Score the held-out edges against an equal batch of fake edges. A well-trained GAE assigns
clearly higher probabilities to the real (held-out) edges.

In [ ]:
z = encode()
test_neg = sample_negatives(len(test_pos))
pos_prob = tf.sigmoid(edge_logits(z, test_pos)).numpy()
neg_prob = tf.sigmoid(edge_logits(z, test_neg)).numpy()
print(f'Final test AUC: {auc_score(pos_prob, neg_prob):.3f}\n')
print(f'Mean prob on REAL held-out edges: {pos_prob.mean():.3f}')
print(f'Mean prob on FAKE edges:          {neg_prob.mean():.3f}')

### Things to try

- Increase the embedding size or add a layer and watch AUC change.
- Swap the dot-product decoder for a small MLP on `[z_i || z_j]`.
- Average AUC over several random edge splits for a more reliable estimate.
- Add a KL term on the embeddings to turn the GAE into a **variational** GAE (VGAE).